Setup + Load ELO

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
from math import exp, factorial

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

elo_files = sorted(RAW_DIR.glob("elo_*_results.csv"))

dfs = []
for file in elo_files:
    temp = pd.read_csv(file)
    temp["source_file"] = file.name
    dfs.append(temp)

elo_df = pd.concat(dfs, ignore_index=True)

print("Loaded ELO shape:", elo_df.shape)
display(elo_df.head())
print(elo_df.columns.tolist())

Loaded ELO shape: (12783, 16)


,date,team_a,team_b,goals_a,goals_b,competition,location,rating_change_a,rating_change_b,rating_a,rating_b,rank_change_a,rank_change_b,rank_a,rank_b,source_file
0,2013-01-05,Angola,Zambia,2,0,Friendly,South Africa,19,-19,1503,1568,6,-5,86,67,elo_2013_results.csv
1,2013-01-05,Bahrain,Oman,0,0,Gulf Cup,Bahrain,-4,4,1493,1527,-3,0,89,80,elo_2013_results.csv
2,2013-01-05,Niger,Togo,3,1,Friendly,Niger,14,-14,1353,1404,1,-2,124,108,elo_2013_results.csv
3,2013-01-05,United Arab Emirates,Qatar,3,1,Gulf Cup,Bahrain,30,-30,1508,1446,10,-3,83,97,elo_2013_results.csv
4,2013-01-06,Iraq,Saudi Arabia,2,0,Gulf Cup,Bahrain,22,-22,1613,1471,6,-4,55,93,elo_2013_results.csv


['date', 'team_a', 'team_b', 'goals_a', 'goals_b', 'competition', 'location', 'rating_change_a', 'rating_change_b', 'rating_a', 'rating_b', 'rank_change_a', 'rank_change_b', 'rank_a', 'rank_b', 'source_file']


 Basic Features 

In [14]:
df = elo_df.copy()

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date", "team_a", "team_b", "goals_a", "goals_b"])
df = df.sort_values("date").reset_index(drop=True)

df["target_goals_a"] = df["goals_a"]
df["target_goals_b"] = df["goals_b"]
df["target_goal_diff"] = df["goals_a"] - df["goals_b"]
df["target_total_goals"] = df["goals_a"] + df["goals_b"]

df["result_a"] = np.where(
    df["goals_a"] > df["goals_b"], 3,
    np.where(df["goals_a"] == df["goals_b"], 1, 0)
)

df["result_b"] = np.where(
    df["goals_b"] > df["goals_a"], 3,
    np.where(df["goals_a"] == df["goals_b"], 1, 0)
)

# rating/rank בדאטה הם אחרי המשחק, לכן מחזירים ללפני המשחק
df["rating_a_before"] = df["rating_a"] - df["rating_change_a"]
df["rating_b_before"] = df["rating_b"] - df["rating_change_b"]

df["rank_a_before"] = df["rank_a"] - df["rank_change_a"]
df["rank_b_before"] = df["rank_b"] - df["rank_change_b"]

df["elo_diff"] = df["rating_a_before"] - df["rating_b_before"]
df["rank_diff"] = df["rank_a_before"] - df["rank_b_before"]

df["is_home_adv"] = (
    df["team_a"].astype(str).str.strip().str.lower()
    ==
    df["location"].astype(str).str.strip().str.lower()
).astype(int)

print("✅ Basic features created")
display(df.head())

✅ Basic features created


,date,team_a,team_b,goals_a,goals_b,competition,location,rating_change_a,rating_change_b,rating_a,...,target_total_goals,result_a,result_b,rating_a_before,rating_b_before,rank_a_before,rank_b_before,elo_diff,rank_diff,is_home_adv
0,2013-01-05,Angola,Zambia,2,0,Friendly,South Africa,19,-19,1503,...,2,3,0,1484,1587,80,72,-103,8,0
1,2013-01-05,Bahrain,Oman,0,0,Gulf Cup,Bahrain,-4,4,1493,...,0,1,1,1497,1523,92,80,-26,12,1
2,2013-01-05,Niger,Togo,3,1,Friendly,Niger,14,-14,1353,...,4,3,0,1339,1418,123,110,-79,13,1
3,2013-01-05,United Arab Emirates,Qatar,3,1,Gulf Cup,Bahrain,30,-30,1508,...,4,3,0,1478,1476,73,100,2,-27,0
4,2013-01-06,Iraq,Saudi Arabia,2,0,Gulf Cup,Bahrain,22,-22,1613,...,2,3,0,1591,1493,49,97,98,-48,0


Rolling Features

In [15]:
WINDOW = 5
team_history = {}

ELO_BASE = df[["rating_a_before", "rating_b_before"]].stack().mean()

features = {
    "team_a_form_last5": [],
    "team_b_form_last5": [],

    "team_a_goals_for_avg_last5": [],
    "team_b_goals_for_avg_last5": [],
    "team_a_goals_against_avg_last5": [],
    "team_b_goals_against_avg_last5": [],

    "team_a_weighted_goals_for_avg_last5": [],
    "team_b_weighted_goals_for_avg_last5": [],
    "team_a_weighted_goals_against_avg_last5": [],
    "team_b_weighted_goals_against_avg_last5": [],

    "team_a_rating_change_avg_last5": [],
    "team_b_rating_change_avg_last5": [],
    "team_a_rank_change_avg_last5": [],
    "team_b_rank_change_avg_last5": [],

    "team_a_matches_played_before": [],
    "team_b_matches_played_before": [],

    "team_a_days_since_last_match": [],
    "team_b_days_since_last_match": [],

    "team_a_win_streak": [],
    "team_b_win_streak": [],
}

def mean_or_zero(values):
    return float(np.mean(values)) if len(values) > 0 else 0.0

def current_win_streak(history):
    streak = 0
    for match in reversed(history):
        if match["points"] == 3:
            streak += 1
        else:
            break
    return streak

for _, row in df.iterrows():
    team_a = row["team_a"]
    team_b = row["team_b"]
    date = row["date"]

    team_history.setdefault(team_a, [])
    team_history.setdefault(team_b, [])

    hist_a_all = team_history[team_a]
    hist_b_all = team_history[team_b]

    hist_a = hist_a_all[-WINDOW:]
    hist_b = hist_b_all[-WINDOW:]

    features["team_a_form_last5"].append(mean_or_zero([x["points"] for x in hist_a]))
    features["team_b_form_last5"].append(mean_or_zero([x["points"] for x in hist_b]))

    features["team_a_goals_for_avg_last5"].append(mean_or_zero([x["goals_for"] for x in hist_a]))
    features["team_b_goals_for_avg_last5"].append(mean_or_zero([x["goals_for"] for x in hist_b]))

    features["team_a_goals_against_avg_last5"].append(mean_or_zero([x["goals_against"] for x in hist_a]))
    features["team_b_goals_against_avg_last5"].append(mean_or_zero([x["goals_against"] for x in hist_b]))

    features["team_a_weighted_goals_for_avg_last5"].append(mean_or_zero([x["weighted_goals_for"] for x in hist_a]))
    features["team_b_weighted_goals_for_avg_last5"].append(mean_or_zero([x["weighted_goals_for"] for x in hist_b]))

    features["team_a_weighted_goals_against_avg_last5"].append(mean_or_zero([x["weighted_goals_against"] for x in hist_a]))
    features["team_b_weighted_goals_against_avg_last5"].append(mean_or_zero([x["weighted_goals_against"] for x in hist_b]))

    features["team_a_rating_change_avg_last5"].append(mean_or_zero([x["rating_change"] for x in hist_a]))
    features["team_b_rating_change_avg_last5"].append(mean_or_zero([x["rating_change"] for x in hist_b]))

    features["team_a_rank_change_avg_last5"].append(mean_or_zero([x["rank_change"] for x in hist_a]))
    features["team_b_rank_change_avg_last5"].append(mean_or_zero([x["rank_change"] for x in hist_b]))

    features["team_a_matches_played_before"].append(len(hist_a_all))
    features["team_b_matches_played_before"].append(len(hist_b_all))

    features["team_a_days_since_last_match"].append(
        (date - hist_a_all[-1]["date"]).days if hist_a_all else 999
    )
    features["team_b_days_since_last_match"].append(
        (date - hist_b_all[-1]["date"]).days if hist_b_all else 999
    )

    features["team_a_win_streak"].append(current_win_streak(hist_a_all))
    features["team_b_win_streak"].append(current_win_streak(hist_b_all))

    opponent_weight_for_a = row["rating_b_before"] / ELO_BASE
    opponent_weight_for_b = row["rating_a_before"] / ELO_BASE

    team_history[team_a].append({
        "date": date,
        "points": row["result_a"],
        "goals_for": row["goals_a"],
        "goals_against": row["goals_b"],
        "weighted_goals_for": row["goals_a"] * opponent_weight_for_a,
        "weighted_goals_against": row["goals_b"] * (ELO_BASE / row["rating_b_before"]),
        "rating_change": row["rating_change_a"],
        "rank_change": row["rank_change_a"],
    })

    team_history[team_b].append({
        "date": date,
        "points": row["result_b"],
        "goals_for": row["goals_b"],
        "goals_against": row["goals_a"],
        "weighted_goals_for": row["goals_b"] * opponent_weight_for_b,
        "weighted_goals_against": row["goals_a"] * (ELO_BASE / row["rating_a_before"]),
        "rating_change": row["rating_change_b"],
        "rank_change": row["rank_change_b"],
    })

for col, values in features.items():
    df[col] = values

print("✅ Rolling + weighted features created")
print("ELO_BASE:", ELO_BASE)
display(df.head(30))

✅ Rolling + weighted features created
ELO_BASE: 1469.9847844793867


,date,team_a,team_b,goals_a,goals_b,competition,location,rating_change_a,rating_change_b,rating_a,...,team_a_rating_change_avg_last5,team_b_rating_change_avg_last5,team_a_rank_change_avg_last5,team_b_rank_change_avg_last5,team_a_matches_played_before,team_b_matches_played_before,team_a_days_since_last_match,team_b_days_since_last_match,team_a_win_streak,team_b_win_streak
0,2013-01-05,Angola,Zambia,2,0,Friendly,South Africa,19,-19,1503,...,0.0,0.0,0.0,0.0,0,0,999,999,0,0
1,2013-01-05,Bahrain,Oman,0,0,Gulf Cup,Bahrain,-4,4,1493,...,0.0,0.0,0.0,0.0,0,0,999,999,0,0
2,2013-01-05,Niger,Togo,3,1,Friendly,Niger,14,-14,1353,...,0.0,0.0,0.0,0.0,0,0,999,999,0,0
3,2013-01-05,United Arab Emirates,Qatar,3,1,Gulf Cup,Bahrain,30,-30,1508,...,0.0,0.0,0.0,0.0,0,0,999,999,0,0
4,2013-01-06,Iraq,Saudi Arabia,2,0,Gulf Cup,Bahrain,22,-22,1613,...,0.0,0.0,0.0,0.0,0,0,999,999,0,0
5,2013-01-06,Kuwait,Yemen,2,0,Gulf Cup,Bahrain,13,-13,1505,...,0.0,0.0,0.0,0.0,0,0,999,999,0,0
6,2013-01-07,Ethiopia,Tunisia,1,1,Friendly,Qatar,7,-7,1325,...,0.0,0.0,0.0,0.0,0,0,999,999,0,0
7,2013-01-08,Bahrain,United Arab Emirates,1,2,Gulf Cup,Bahrain,-25,25,1468,...,-4.0,30.0,-3.0,10.0,1,1,3,3,0,1
8,2013-01-08,Morocco,Zambia,0,0,Friendly,South Africa,1,-1,1544,...,0.0,-19.0,0.0,-5.0,0,1,999,3,0,0
9,2013-01-08,Qatar,Oman,2,1,Gulf Cup,Bahrain,25,-25,1471,...,-30.0,4.0,-3.0,0.0,1,1,3,3,0,0


 Diff Features + Save

In [16]:
df["form_diff_last5"] = df["team_a_form_last5"] - df["team_b_form_last5"]

df["goals_for_diff_last5"] = df["team_a_goals_for_avg_last5"] - df["team_b_goals_for_avg_last5"]
df["goals_against_diff_last5"] = df["team_a_goals_against_avg_last5"] - df["team_b_goals_against_avg_last5"]

df["weighted_goals_for_diff_last5"] = (
    df["team_a_weighted_goals_for_avg_last5"] -
    df["team_b_weighted_goals_for_avg_last5"]
)

df["weighted_goals_against_diff_last5"] = (
    df["team_a_weighted_goals_against_avg_last5"] -
    df["team_b_weighted_goals_against_avg_last5"]
)

df["rating_change_diff_last5"] = df["team_a_rating_change_avg_last5"] - df["team_b_rating_change_avg_last5"]
df["rank_change_diff_last5"] = df["team_a_rank_change_avg_last5"] - df["team_b_rank_change_avg_last5"]

df["days_since_match_diff"] = df["team_a_days_since_last_match"] - df["team_b_days_since_last_match"]
df["win_streak_diff"] = df["team_a_win_streak"] - df["team_b_win_streak"]

model_df = df[
    [
        "date",
        "team_a",
        "team_b",
        "competition",
        "location",
        "is_home_adv",

        "rating_a_before",
        "rating_b_before",
        "elo_diff",

        "rank_a_before",
        "rank_b_before",
        "rank_diff",

        "team_a_form_last5",
        "team_b_form_last5",
        "form_diff_last5",

        "team_a_goals_for_avg_last5",
        "team_b_goals_for_avg_last5",
        "goals_for_diff_last5",

        "team_a_goals_against_avg_last5",
        "team_b_goals_against_avg_last5",
        "goals_against_diff_last5",

        "team_a_weighted_goals_for_avg_last5",
        "team_b_weighted_goals_for_avg_last5",
        "weighted_goals_for_diff_last5",

        "team_a_weighted_goals_against_avg_last5",
        "team_b_weighted_goals_against_avg_last5",
        "weighted_goals_against_diff_last5",

        "team_a_rating_change_avg_last5",
        "team_b_rating_change_avg_last5",
        "rating_change_diff_last5",

        "team_a_rank_change_avg_last5",
        "team_b_rank_change_avg_last5",
        "rank_change_diff_last5",

        "team_a_matches_played_before",
        "team_b_matches_played_before",

        "team_a_days_since_last_match",
        "team_b_days_since_last_match",
        "days_since_match_diff",

        "team_a_win_streak",
        "team_b_win_streak",
        "win_streak_diff",

        "target_goals_a",
        "target_goals_b",
        "target_goal_diff",
        "target_total_goals",
    ]
].copy()

output_path = PROCESSED_DIR / "elo_model_dataset.csv"
model_df.to_csv(output_path, index=False)

print("✅ Model dataset saved")
print("Shape:", model_df.shape)
print("Saved to:", output_path)

display(model_df.head())

✅ Model dataset saved
Shape: (12783, 45)
Saved to: ../data/processed/elo_model_dataset.csv


,date,team_a,team_b,competition,location,is_home_adv,rating_a_before,rating_b_before,elo_diff,rank_a_before,...,team_a_days_since_last_match,team_b_days_since_last_match,days_since_match_diff,team_a_win_streak,team_b_win_streak,win_streak_diff,target_goals_a,target_goals_b,target_goal_diff,target_total_goals
0,2013-01-05,Angola,Zambia,Friendly,South Africa,0,1484,1587,-103,80,...,999,999,0,0,0,0,2,0,2,2
1,2013-01-05,Bahrain,Oman,Gulf Cup,Bahrain,1,1497,1523,-26,92,...,999,999,0,0,0,0,0,0,0,0
2,2013-01-05,Niger,Togo,Friendly,Niger,1,1339,1418,-79,123,...,999,999,0,0,0,0,3,1,2,4
3,2013-01-05,United Arab Emirates,Qatar,Gulf Cup,Bahrain,0,1478,1476,2,73,...,999,999,0,0,0,0,3,1,2,4
4,2013-01-06,Iraq,Saudi Arabia,Gulf Cup,Bahrain,0,1591,1493,98,49,...,999,999,0,0,0,0,2,0,2,2


Train Model + Filter

In [17]:
model_df = pd.read_csv("../data/processed/elo_model_dataset.csv")
model_df["date"] = pd.to_datetime(model_df["date"])
model_df = model_df.sort_values("date").reset_index(drop=True)

MIN_RATING = 1500

model_df_train = model_df[
    (model_df["rating_a_before"] >= MIN_RATING) &
    (model_df["rating_b_before"] >= MIN_RATING)
].copy()

model_df_train["days_since_match_diff"] = model_df_train["days_since_match_diff"].clip(-60, 60)

model_df_train = model_df_train.sort_values("date").reset_index(drop=True)

print("Before filter:", model_df.shape)
print("After filter:", model_df_train.shape)

feature_cols = [
    "is_home_adv",

    "rating_a_before",
    "rating_b_before",
    "elo_diff",

    "rank_a_before",
    "rank_b_before",
    "rank_diff",

    "team_a_form_last5",
    "team_b_form_last5",
    "form_diff_last5",

    "team_a_goals_for_avg_last5",
    "team_b_goals_for_avg_last5",
    "goals_for_diff_last5",

    "team_a_goals_against_avg_last5",
    "team_b_goals_against_avg_last5",
    "goals_against_diff_last5",

    "team_a_weighted_goals_for_avg_last5",
    "team_b_weighted_goals_for_avg_last5",
    "weighted_goals_for_diff_last5",

    "team_a_weighted_goals_against_avg_last5",
    "team_b_weighted_goals_against_avg_last5",
    "weighted_goals_against_diff_last5",

    "team_a_rating_change_avg_last5",
    "team_b_rating_change_avg_last5",
    "rating_change_diff_last5",

    "team_a_rank_change_avg_last5",
    "team_b_rank_change_avg_last5",
    "rank_change_diff_last5",

    "team_a_matches_played_before",
    "team_b_matches_played_before",

    "team_a_days_since_last_match",
    "team_b_days_since_last_match",
    "days_since_match_diff",

    "team_a_win_streak",
    "team_b_win_streak",
    "win_streak_diff",
]

target_cols = ["target_goals_a", "target_goals_b"]

X = model_df_train[feature_cols]
y = model_df_train[target_cols]

split_idx = int(len(model_df_train) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=500,
        max_depth=10,
        min_samples_leaf=8,
        random_state=42,
        n_jobs=-1
    )
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

mse = mean_squared_error(y_test, preds)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, preds)

print("Regression metrics")
print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)

pred_df = model_df_train.iloc[split_idx:][
    ["date", "team_a", "team_b", "target_goals_a", "target_goals_b"]
].copy()

pred_df["pred_goals_a"] = preds[:, 0]
pred_df["pred_goals_b"] = preds[:, 1]

display(pred_df.head(30))

Before filter: (12783, 45)
After filter: (4486, 45)
Train: (3588, 36) (3588, 2)
Test: (898, 36) (898, 2)
Regression metrics
MSE: 0.9970650137922292
RMSE: 0.9985314285450555
MAE: 0.7736365334619435


,date,team_a,team_b,target_goals_a,target_goals_b,pred_goals_a,pred_goals_b
3588,2023-09-07,Norway,Jordan,6,0,1.825137,0.442286
3589,2023-09-07,Czechia,Albania,1,1,1.652969,0.645017
3590,2023-09-08,Brazil,Bolivia,5,1,2.958738,0.451246
3591,2023-09-08,Uruguay,Chile,3,1,1.894373,0.491580
3592,2023-09-08,Costa Rica,Saudi Arabia,3,1,1.737942,0.687324
3593,2023-09-08,Slovakia,Portugal,0,1,0.502986,2.056896
3594,2023-09-08,Georgia,Spain,1,7,0.624120,1.694614
3595,2023-09-09,United States,Uzbekistan,3,0,2.308537,0.542684
3596,2023-09-09,Australia,Mexico,2,2,1.432370,1.023888
3597,2023-09-09,Germany,Japan,1,4,1.075272,1.874429


Poisson Exact Score

In [18]:
def poisson_prob(lam, k):
    return (lam ** k) * exp(-lam) / factorial(k)

def predict_score_probs(lambda_a, lambda_b, max_goals=6):
    scores = []

    for goals_a in range(max_goals + 1):
        for goals_b in range(max_goals + 1):
            prob = poisson_prob(lambda_a, goals_a) * poisson_prob(lambda_b, goals_b)

            scores.append({
                "pred_score": f"{goals_a}-{goals_b}",
                "goals_a": goals_a,
                "goals_b": goals_b,
                "probability": prob
            })

    return pd.DataFrame(scores).sort_values("probability", ascending=False)

poisson_results = []

for _, row in pred_df.iterrows():
    lambda_a = min(max(row["pred_goals_a"], 0.05), 4.0)
    lambda_b = min(max(row["pred_goals_b"], 0.05), 4.0)

    score_probs = predict_score_probs(lambda_a, lambda_b, max_goals=6)
    best_score = score_probs.iloc[0]

    poisson_results.append({
        "date": row["date"],
        "team_a": row["team_a"],
        "team_b": row["team_b"],
        "actual_score": f"{int(row['target_goals_a'])}-{int(row['target_goals_b'])}",
        "pred_score": best_score["pred_score"],
        "pred_score_probability": best_score["probability"],
        "lambda_a": lambda_a,
        "lambda_b": lambda_b
    })

poisson_pred_df = pd.DataFrame(poisson_results)

poisson_pred_df["exact_score_hit"] = (
    poisson_pred_df["actual_score"] == poisson_pred_df["pred_score"]
).astype(int)

exact_score_accuracy = poisson_pred_df["exact_score_hit"].mean()

print("Poisson exact score metrics")
print("Exact score accuracy:", exact_score_accuracy)
print("Exact score accuracy %:", round(exact_score_accuracy * 100, 2))

display(poisson_pred_df.head(50))

Poisson exact score metrics
Exact score accuracy: 0.21158129175946547
Exact score accuracy %: 21.16


,date,team_a,team_b,actual_score,pred_score,pred_score_probability,lambda_a,lambda_b,exact_score_hit
0,2023-09-07,Norway,Jordan,6-0,1-0,0.189045,1.825137,0.442286,0
1,2023-09-07,Czechia,Albania,1-1,1-0,0.166059,1.652969,0.645017,0
2,2023-09-08,Brazil,Bolivia,5-1,2-0,0.144626,2.958738,0.451246,0
3,2023-09-08,Uruguay,Chile,3-1,1-0,0.174285,1.894373,0.491580,0
4,2023-09-08,Costa Rica,Saudi Arabia,3-1,1-0,0.153729,1.737942,0.687324,0
5,2023-09-08,Slovakia,Portugal,0-1,0-2,0.163551,0.502986,2.056896,0
6,2023-09-08,Georgia,Spain,1-7,0-1,0.166747,0.624120,1.694614,0
7,2023-09-09,United States,Uzbekistan,3-0,2-0,0.153948,2.308537,0.542684,0
8,2023-09-09,Australia,Mexico,2-2,1-1,0.125767,1.432370,1.023888,0
9,2023-09-09,Germany,Japan,1-4,1-1,0.105523,1.075272,1.874429,0


Feature Importance

In [19]:
importances_a = model.estimators_[0].feature_importances_
importances_b = model.estimators_[1].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_goals_a": importances_a,
    "importance_goals_b": importances_b,
})

importance_df["avg_importance"] = (
    importance_df["importance_goals_a"] + importance_df["importance_goals_b"]
) / 2

importance_df = importance_df.sort_values("avg_importance", ascending=False)

display(importance_df)

,feature,importance_goals_a,importance_goals_b,avg_importance
6,rank_diff,0.438756,0.428732,0.433744
3,elo_diff,0.099129,0.107329,0.103229
1,rating_a_before,0.028156,0.096972,0.062564
2,rating_b_before,0.058808,0.020919,0.039864
4,rank_a_before,0.031459,0.026054,0.028756
20,team_b_weighted_goals_against_avg_last5,0.021071,0.020907,0.020989
5,rank_b_before,0.016265,0.024295,0.020280
16,team_a_weighted_goals_for_avg_last5,0.018711,0.016695,0.017703
19,team_a_weighted_goals_against_avg_last5,0.018085,0.017160,0.017623
18,weighted_goals_for_diff_last5,0.018265,0.016450,0.017357


Correlation Matrix

In [20]:
# ==============================
# Smart feature pruning
# Remove ONLY weak + correlated features
# ==============================

CORR_THRESHOLD = 0.85
MIN_IMPORTANCE = 0.01

corr_matrix = model_df_train[feature_cols].corr().abs()

importance_map = importance_df.set_index("feature")["avg_importance"].to_dict()

features_to_drop = set()
drop_decisions = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):

        f1 = corr_matrix.columns[i]
        f2 = corr_matrix.columns[j]

        corr_value = corr_matrix.iloc[i, j]

        if corr_value >= CORR_THRESHOLD:

            imp1 = importance_map.get(f1, 0)
            imp2 = importance_map.get(f2, 0)

            # מוחקים רק אם אחד מהם באמת חלש
            if imp1 < MIN_IMPORTANCE or imp2 < MIN_IMPORTANCE:

                if imp1 >= imp2:
                    keep_feature = f1
                    drop_feature = f2
                else:
                    keep_feature = f2
                    drop_feature = f1

                features_to_drop.add(drop_feature)

                drop_decisions.append({
                    "feature_1": f1,
                    "feature_2": f2,
                    "correlation": corr_value,
                    "importance_1": imp1,
                    "importance_2": imp2,
                    "keep": keep_feature,
                    "drop": drop_feature
                })

smart_drop_df = pd.DataFrame(drop_decisions).sort_values(
    "correlation",
    ascending=False
)

smart_selected_features = [
    f for f in feature_cols
    if f not in features_to_drop
]

print("Original features:", len(feature_cols))
print("Dropped features:", len(features_to_drop))
print("Remaining features:", len(smart_selected_features))

display(smart_drop_df)

print("\nSelected features:")
print(smart_selected_features)

Original features: 36
Dropped features: 8
Remaining features: 28


,feature_1,feature_2,correlation,importance_1,importance_2,keep,drop
5,goals_against_diff_last5,weighted_goals_against_diff_last5,0.986100,0.003772,0.015996,weighted_goals_against_diff_last5,goals_against_diff_last5
3,team_a_goals_against_avg_last5,team_a_weighted_goals_against_avg_last5,0.982997,0.003557,0.017623,team_a_weighted_goals_against_avg_last5,team_a_goals_against_avg_last5
4,team_b_goals_against_avg_last5,team_b_weighted_goals_against_avg_last5,0.980914,0.003762,0.020989,team_b_weighted_goals_against_avg_last5,team_b_goals_against_avg_last5
2,goals_for_diff_last5,weighted_goals_for_diff_last5,0.962076,0.006326,0.017357,weighted_goals_for_diff_last5,goals_for_diff_last5
1,team_b_goals_for_avg_last5,team_b_weighted_goals_for_avg_last5,0.952729,0.006337,0.016215,team_b_weighted_goals_for_avg_last5,team_b_goals_for_avg_last5
0,team_a_goals_for_avg_last5,team_a_weighted_goals_for_avg_last5,0.950880,0.005474,0.017703,team_a_weighted_goals_for_avg_last5,team_a_goals_for_avg_last5
6,team_b_rating_change_avg_last5,team_b_rank_change_avg_last5,0.863628,0.015620,0.008090,team_b_rating_change_avg_last5,team_b_rank_change_avg_last5
7,rating_change_diff_last5,rank_change_diff_last5,0.854908,0.015024,0.009646,rating_change_diff_last5,rank_change_diff_last5



Selected features:
['is_home_adv', 'rating_a_before', 'rating_b_before', 'elo_diff', 'rank_a_before', 'rank_b_before', 'rank_diff', 'team_a_form_last5', 'team_b_form_last5', 'form_diff_last5', 'team_a_weighted_goals_for_avg_last5', 'team_b_weighted_goals_for_avg_last5', 'weighted_goals_for_diff_last5', 'team_a_weighted_goals_against_avg_last5', 'team_b_weighted_goals_against_avg_last5', 'weighted_goals_against_diff_last5', 'team_a_rating_change_avg_last5', 'team_b_rating_change_avg_last5', 'rating_change_diff_last5', 'team_a_rank_change_avg_last5', 'team_a_matches_played_before', 'team_b_matches_played_before', 'team_a_days_since_last_match', 'team_b_days_since_last_match', 'days_since_match_diff', 'team_a_win_streak', 'team_b_win_streak', 'win_streak_diff']


Lean Model

In [21]:
# ==============================
# Train Smart-Pruned Model
# ==============================

X = model_df_train[smart_selected_features]
y = model_df_train[target_cols]

split_idx = int(len(model_df_train) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

smart_model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=500,
        max_depth=10,
        min_samples_leaf=8,
        random_state=42,
        n_jobs=-1
    )
)

smart_model.fit(X_train, y_train)

smart_preds = smart_model.predict(X_test)

smart_mse = mean_squared_error(y_test, smart_preds)
smart_rmse = np.sqrt(smart_mse)
smart_mae = mean_absolute_error(y_test, smart_preds)

print("Smart-Pruned Regression metrics")
print("MSE:", smart_mse)
print("RMSE:", smart_rmse)
print("MAE:", smart_mae)

Smart-Pruned Regression metrics
MSE: 0.9986464812896929
RMSE: 0.9993230114881239
MAE: 0.7736126196994779


Poisson to Lean Model

In [22]:
# ==============================
# Poisson exact score for Smart-Pruned Model
# ==============================

smart_pred_df = model_df_train.iloc[split_idx:][
    ["date", "team_a", "team_b", "target_goals_a", "target_goals_b"]
].copy()

smart_pred_df["pred_goals_a"] = smart_preds[:, 0]
smart_pred_df["pred_goals_b"] = smart_preds[:, 1]

smart_poisson_results = []

for _, row in smart_pred_df.iterrows():

    lambda_a = min(max(row["pred_goals_a"], 0.05), 4.0)
    lambda_b = min(max(row["pred_goals_b"], 0.05), 4.0)

    score_probs = predict_score_probs(lambda_a, lambda_b, max_goals=6)

    best_score = score_probs.iloc[0]

    smart_poisson_results.append({
        "actual_score": f"{int(row['target_goals_a'])}-{int(row['target_goals_b'])}",
        "pred_score": best_score["pred_score"]
    })

smart_poisson_df = pd.DataFrame(smart_poisson_results)

smart_poisson_df["exact_hit"] = (
    smart_poisson_df["actual_score"] ==
    smart_poisson_df["pred_score"]
).astype(int)

smart_exact_acc = smart_poisson_df["exact_hit"].mean()

print("\nSmart-Pruned Exact Score Accuracy")
print("Accuracy:", smart_exact_acc)
print("Accuracy %:", round(smart_exact_acc * 100, 2))


Smart-Pruned Exact Score Accuracy
Accuracy: 0.20824053452115812
Accuracy %: 20.82
